In [ ]:
!pip install python-dotenv networkx neo4j==5.23.0 graspologic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 45.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 39.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.6/865.6 kB 24.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [136]:
import os
import logging
import neo4j
from neo4j import Driver
import networkx as nx
import time
from graspologic.partition import hierarchical_leiden

In [ ]:
%env NEO4J_URI=neo4j://10.88.203.244:7687
%env NEO4J_PASSWORD=
%env NEO4J_USERNAME=neo4j

In [138]:
logging.basicConfig(level=logging.INFO)

logger = logging.getLogger(__name__)

In [139]:
def _driver() -> Driver:
    neo4j_uri = os.environ["NEO4J_URI"]
    neo4j_user = os.environ["NEO4J_USERNAME"]
    neo4j_passwd = os.environ["NEO4J_PASSWORD"]
    return neo4j.GraphDatabase.driver(
        uri=neo4j_uri, auth=(neo4j_user, neo4j_passwd)
    )

In [140]:
def _fetch_all(limit=100):
    query = f"MATCH (n)-[r]->(c) RETURN * LIMIT {limit}"
    results = _driver().session().run(query)
    logger.debug(f"results: {results}")
    graph = nx.Graph()
    nodes = list(results.graph()._nodes.values())
    for node in nodes:
        graph.add_node(node.id, labels=node._labels, properties=node._properties)
    rels = list(results.graph()._relationships.values())
    for rel in rels:
        graph.add_edge(
            rel.start_node.id,
            rel.end_node.id,
            key=rel.id,
            type=rel.type,
            properties=rel._properties
        )

    logger.info(f"{graph}")
    # logger.debug(f"{graph.nodes()}")
    # logger.debug(f"{graph.edges()}")
    # logger.debug("nodes")
    # for node in graph.nodes.data():
    #     logger.debug(f"{node}")

    # logger.debug("relationships")
    # for edge in graph.edges.data():
    #     logger.debug(f"{edge}")
    return graph

In [141]:
def _collect_community_info(nx_graph, clusters) -> dict[str:dict[str:str]]:
    """
    Collect detailed information for each node based on their community.
    """
    community_mapping = {item.node: item.cluster for item in clusters}
    community_info = {}
    for item in clusters:
        cluster_id = item.cluster
        # parent_cluster_id = item.parent_cluster or 0
        # composite_id = f"{parent_cluster_id:03}-{cluster_id:03}"
        composite_id = cluster_id
        node = item.node
        if composite_id not in community_info:
            logger.debug(f"new cluster_id {cluster_id}")
            community_info[composite_id] = []

        for neighbor in nx_graph.neighbors(node):
            if community_mapping[neighbor] == cluster_id:
                edge_data = nx_graph.get_edge_data(node, neighbor)
                if edge_data:
                    detail = {
                        "source": node,
                        "target": neighbor,
                        "data": edge_data
                    }
                    # detail = f"{node}:{neighbor} -> {edge_data}"
                    community_info[composite_id].append(detail)
    return community_info

In [142]:
def _summarize_communities(community_info):
    count = 0
    current_total = 0
    for key in community_info.keys():
        count += 1
        current_total += len(community_info[key])
    avg = current_total / count
    logger.info(f"{count} communities with avg community size of {avg}")

In [143]:
def _rank_and_print_communities(community_info):
    for w in sorted(community_info, key=lambda x: len(community_info[x]), reverse=True):
        print(w, len(community_info[w]))

In [149]:
def _collect_index_ids(graph: nx.Graph, community: list[dict[str: any]]):
    uniq_index_ids = set()
    for relation in community:
        source = graph.nodes[relation["source"]]
        target = graph.nodes[relation["target"]]
        source_index = source["properties"]["node_index"]
        target_index = target["properties"]["node_index"]
        uniq_index_ids.add(source_index)
        uniq_index_ids.add(target_index)
    return uniq_index_ids

In [150]:
def _generate_community_query(graph: nx.Graph, community: dict[str: dict[str: any]]):
    uniq_index_ids = _collect_index_ids(graph, community)
    logger.info(f"{uniq_index_ids}")
    ids = ','.join([f"'{el}'" for el in uniq_index_ids])
    query = f"""
    WITH [{ids}] AS ids
    UNWIND ids AS id
    MATCH (n)
    WHERE n.node_index = id
    RETURN n
    """
    return query

In [151]:
start = time.perf_counter()
limit = 1_000_000
graph = _fetch_all(limit)
end = time.perf_counter()
logger.info(f"loaded limit {limit} in {end - start:0.4f} seconds")
logger.info(f"{graph}")

/tmp/ipykernel_9400/645202464.py:8: DeprecationWarning: `id` is deprecated, use `element_id` instead
  graph.add_node(node.id, labels=node._labels, properties=node._properties)
/tmp/ipykernel_9400/645202464.py:12: DeprecationWarning: `id` is deprecated, use `element_id` instead
  rel.start_node.id,
/tmp/ipykernel_9400/645202464.py:13: DeprecationWarning: `id` is deprecated, use `element_id` instead
  rel.end_node.id,
/tmp/ipykernel_9400/645202464.py:14: DeprecationWarning: `id` is deprecated, use `element_id` instead
  key=rel.id,
INFO:__main__:Graph with 28325 nodes and 999752 edges
INFO:__main__:loaded limit 1000000 in 210.6058 seconds
INFO:__main__:Graph with 28325 nodes and 999752 edges


In [152]:
start = time.perf_counter()
community_hierarchical_clusters = hierarchical_leiden(
    graph, max_cluster_size=75
)
end = time.perf_counter()
logger.info(f"community detection took {end - start:0.4f} seconds")

INFO:__main__:community detection took 11.8343 seconds


In [153]:
logger.info(f"{len(community_hierarchical_clusters)}")
final_clusters = community_hierarchical_clusters.final_level_hierarchical_clustering()
logger.info(f"{len(final_clusters)}")
# logger.info(f"{final_clusters}")
logger.info(f"{community_hierarchical_clusters[64]}")

INFO:__main__:121475
INFO:__main__:28325
INFO:__main__:HierarchicalCluster(node=729, cluster=1, parent_cluster=None, level=0, is_final_cluster=False)


In [154]:
community_info = _collect_community_info(
    graph, community_hierarchical_clusters
)
logger.info(f"{len(community_info)}")

INFO:__main__:1588


In [155]:
_summarize_communities(community_info)

INFO:__main__:1588 communities with avg community size of 154.50251889168766


In [ ]:
_rank_and_print_communities(community_info)

275 65208
1394 22110
1375 8382
1414 6748
1556 5492
1507 3172
1544 2538
1586 2010
1534 1976
1554 1690
363 1646
1456 1552
346 1454
721 1332
1508 1304
1583 1206
1474 1096
234 1032
1533 990
1451 970
147 944
842 888
1450 888
351 816
550 808
111 804
707 774
834 774
807 742
850 712
836 702
827 700
1575 682
180 672
341 664
1581 642
1177 600
348 590
392 590
821 580
380 552
548 548
1481 538
547 530
639 526
685 526
1417 526
378 516
630 504
1526 504
805 502
1514 496
202 476
1499 472
360 470
151 462
1547 442
1368 434
1524 416
1551 416
1267 410
873 400
983 392
674 388
1393 386
289 380
307 376
797 374
581 372
680 372
727 368
123 366
562 364
1505 362
187 360
672 356
1428 354
1116 350
645 346
806 344
1436 338
1118 332
311 326
367 326
608 326
1069 326
313 324
395 318
661 316
165 314
541 314
1187 314
1173 310
393 308
463 308
686 304
677 302
130 300
1416 298
573 294
716 294
762 294
929 294
1468 294
168 290
740 288
131 286
838 284
1503 284
308 282
355 282
528 282
150 280
1513 280
965 278
369 274
587 274
14

In [157]:
ids = [6671, 17369, 3527]
for id in ids:
    logger.info(f"{graph.nodes[id]}")

INFO:__main__:{'labels': frozenset({'gene_protein'}), 'properties': {'node_source': 'NCBI', 'node_name': 'LRP2', 'node_index': '3941', 'node_id': '4036'}}
INFO:__main__:{'labels': frozenset({'gene_protein'}), 'properties': {'node_source': 'NCBI', 'node_name': 'EGFL6', 'node_index': '10817', 'node_id': '25975'}}
INFO:__main__:{'labels': frozenset({'gene_protein'}), 'properties': {'node_source': 'NCBI', 'node_name': 'AMBN', 'node_index': '1889', 'node_id': '258'}}


In [ ]:
query = _generate_community_query(graph, community_info[307])
logger.info(f"{query}")

INFO:__main__:{'5793', '60231', '5376', '19592', '12050', '12481', '57167', '513', '4749', '4325', '5715', '8656', '5621', '11357', '9183', '12723', '9345', '8234', '6184', '4821', '57622', '914', '56797', '3968', '3341', '1464', '8871', '5532', '13540', '10974', '1653', '3927', '514', '12557', '7070'}
INFO:__main__:
    WITH ['5793','60231','5376','19592','12050','12481','57167','513','4749','4325','5715','8656','5621','11357','9183','12723','9345','8234','6184','4821','57622','914','56797','3968','3341','1464','8871','5532','13540','10974','1653','3927','514','12557','7070'] AS ids
    UNWIND ids AS id
    MATCH (n)
    WHERE n.node_index = id
    RETURN n
    


In [161]:
len(community_info[307])

376